In [213]:
# Step 1: Generate Shapes Dataset (Ensures COCO Format)
import subprocess
import os

# Define dataset path
dataset_path = "./shapes_dataset"

# Set dynamic parameters
num_images = 5  # Change this dynamically if needed
image_width = 128  # Change this dynamically if needed
image_height = 128  # Change this dynamically if needed

# Run the dataset generation script inside Jupyter
command = [
    "python", "shapes/run.py",
    "--save_dir", dataset_path,
    "--num_images", str(num_images),
    "--image_size", str(image_width), str(image_height),  # ✅ Explicitly setting image size dynamically
    "--task_type", "segmentation"
]

# Execute the command
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)  # Print output for debugging

# Check if dataset is created
if not os.path.exists(f"{dataset_path}/annotations/coco_annotations.json"):
    print("❌ ERROR: Dataset not generated correctly! Check run.py.")
else:
    print(f"✅ Shapes dataset successfully generated with {num_images} images of size {image_width}x{image_height}!")

✅ Dataset and COCO annotations saved to ./shapes_dataset

✅ Shapes dataset successfully generated with 5 images of size 128x128!


In [215]:
#Step 2: Load and Verify COCO Annotations
import json

# Define the correct path for the COCO annotations file
OUTPUT_ANNOTATION_FILE = "./shapes_dataset/annotations/coco_annotations.json"

# Verify COCO Annotations
try:
    with open(OUTPUT_ANNOTATION_FILE, "r") as f:
        coco_data = json.load(f)

    # Show structure
    print("✅ COCO Annotations Loaded Successfully!")
    print(json.dumps(coco_data, indent=4)[:1000])  # Print first 1000 characters

except FileNotFoundError:
    print(f"❌ ERROR: The file {OUTPUT_ANNOTATION_FILE} was not found. Make sure it was generated correctly!")

✅ COCO Annotations Loaded Successfully!
{
    "images": [
        {
            "id": 0,
            "width": 128,
            "height": 128,
            "file_name": "shapes_0.png"
        },
        {
            "id": 1,
            "width": 128,
            "height": 128,
            "file_name": "shapes_1.png"
        },
        {
            "id": 2,
            "width": 128,
            "height": 128,
            "file_name": "shapes_2.png"
        },
        {
            "id": 3,
            "width": 128,
            "height": 128,
            "file_name": "shapes_3.png"
        },
        {
            "id": 4,
            "width": 128,
            "height": 128,
            "file_name": "shapes_4.png"
        }
    ],
    "annotations": [
        {
            "id": 1,
            "image_id": 0,
            "category_id": 2,
            "bbox": [
                26,
                77,
                4,
                4
            ],
            "segmentation": [],
      

In [ ]:
#Step 3
import os
import json
import torch
import numpy as np
import cv2
from segment_anything import sam_model_registry, SamPredictor

# ============================
# 🔹 Load SAM Model with Debugging
# ============================

CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
WEIGHTS_DIR = "/Users/ma/Documents/samThesis/segment-anything/weights"
CHECKPOINT_PATH = os.path.join(WEIGHTS_DIR, "sam_vit_h_4b8939.pth")

# Create directory if needed
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Download checkpoint if not exists
if not os.path.isfile(CHECKPOINT_PATH):
    os.system(f"curl -o {CHECKPOINT_PATH} {CHECKPOINT_URL}")
    print(f"✅ Downloaded SAM checkpoint to: {CHECKPOINT_PATH}")
else:
    print(f"✅ Checkpoint already exists at: {CHECKPOINT_PATH}")

# Load SAM model
sam = sam_model_registry["vit_h"](checkpoint=CHECKPOINT_PATH)
predictor = SamPredictor(sam)

# ============================
# 🔹 Load COCO Annotations with Debugging
# ============================

DATASET_PATH = "./shapes_dataset"
ANNOTATION_FILE = os.path.join(DATASET_PATH, "annotations", "coco_annotations.json")
OUTPUT_ANNOTATION_FILE = os.path.join(DATASET_PATH, "annotations", "coco_annotations_with_masks.json")

# Check if annotation file exists
if not os.path.exists(ANNOTATION_FILE):
    print(f"❌ ERROR: COCO annotation file not found: {ANNOTATION_FILE}")
    raise FileNotFoundError(f"COCO annotation file {ANNOTATION_FILE} does not exist!")

# Load COCO JSON
with open(ANNOTATION_FILE, "r") as f:
    coco_data = json.load(f)

print(f"✅ Loaded COCO annotations from {ANNOTATION_FILE}")
print(f"🔹 Total Images: {len(coco_data['images'])} | Total Annotations: {len(coco_data['annotations'])}")

# Debug: Check if images exist
if len(coco_data["images"]) == 0:
    print("⚠️ WARNING: No images found in COCO annotations! Check dataset generation.")
    raise ValueError("COCO dataset has no images!")

# Debug: Check if annotations exist
if len(coco_data["annotations"]) == 0:
    print("⚠️ WARNING: No annotations found in COCO annotations! Expected bounding boxes.")
    raise ValueError("COCO dataset has no annotations!")

# ============================
# 🔹 Run SAM on Each Image & Generate Masks
# ============================

new_annotations = []
masks_generated = 0
no_masks_found = 0

for img_info in coco_data["images"]:
    img_path = os.path.join(DATASET_PATH, "images", img_info["file_name"])

    # Check if image exists
    if not os.path.exists(img_path):
        print(f"⚠️ Skipping: {img_info['file_name']} (File Not Found)")
        continue

    # Load image
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    predictor.set_image(image)

    print(f"🔹 Running SAM on {img_info['file_name']}...")

    # Get annotations for the current image
    img_annotations = [ann for ann in coco_data["annotations"] if ann["image_id"] == img_info["id"]]

    for ann in img_annotations:
        bbox = ann["bbox"]
        x, y, w, h = map(int, bbox)

        # Create bounding box as input for SAM
        input_box = np.array([x, y, x + w, y + h])
        sam_masks, _, _ = predictor.predict(box=input_box)

        # Debugging: Check if SAM generated masks
        if len(sam_masks) == 0:
            print(f"⚠️ No masks found for {img_info['file_name']}! Skipping annotation.")
            no_masks_found += 1
            continue
        else:
            print(f"✅ {len(sam_masks)} masks detected.")

        # Convert mask to COCO segmentation format
        for mask in sam_masks:
            mask_indices = np.argwhere(mask)  # Get mask indices
            segmentation = mask_indices.flatten().tolist()  # Convert to list
            if segmentation:  # Only add if mask is not empty
                new_annotations.append({
                    "id": len(new_annotations) + 1,
                    "image_id": img_info["id"],
                    "category_id": ann["category_id"],
                    "segmentation": [segmentation],  # List of coordinates
                    "area": int(np.sum(mask)),  # Mask area
                    "bbox": ann["bbox"],  # Keep original bbox
                    "iscrowd": 0
                })
                masks_generated += 1

# ============================
# 🔹 Save Updated COCO Annotations
# ============================

# Update annotations in COCO JSON
coco_data["annotations"] = new_annotations

# Save updated COCO JSON
with open(OUTPUT_ANNOTATION_FILE, "w") as f:
    json.dump(coco_data, f, indent=4)

# Debugging Summary
print("\n✅ SAM Masks Generated and Saved in COCO Format!")
print(f"🔹 Images Processed: {len(coco_data['images'])} | Masks Generated: {masks_generated} | No Masks: {no_masks_found}")

# Final validation check
if len(new_annotations) == 0:
    print("❌ ERROR: No segmentation data generated! Check SAM mask outputs.")

✅ Checkpoint already exists at: /Users/ma/Documents/samThesis/segment-anything/weights/sam_vit_h_4b8939.pth
✅ Loaded COCO annotations from ./shapes_dataset/annotations/coco_annotations.json
🔹 Total Images: 5 | Total Annotations: 7
🔹 Running SAM on shapes_0.png...
✅ 3 masks detected.
🔹 Running SAM on shapes_1.png...
✅ 3 masks detected.
🔹 Running SAM on shapes_2.png...
✅ 3 masks detected.
✅ 3 masks detected.


In [ ]:
# Step 4: Verify Updated COCO Annotations
# Load updated COCO annotations
with open(OUTPUT_ANNOTATION_FILE, "r") as f:
    updated_coco_data = json.load(f)

# Check if segmentation is now included
sample_ann = next((ann for ann in updated_coco_data["annotations"] if "segmentation" in ann), None)

if sample_ann:
    print("✅ Segmentation Data Found in COCO Format!")
    print(json.dumps(sample_ann, indent=4))
else:
    print("❌ ERROR: No segmentation data found. Ensure SAM generated masks correctly!")


In [ ]:
# Step 5: Visualize the Segmentation Masks
import matplotlib.pyplot as plt

# Pick a random image
image_id = np.random.choice(len(updated_coco_data["images"]))
image_info = updated_coco_data["images"][image_id]
image_path = os.path.join(image_dir, image_info["file_name"])

# Load image
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Load segmentation masks
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image)

for ann in updated_coco_data["annotations"]:
    if ann["image_id"] == image_info["id"]:
        for seg in ann["segmentation"]:
            poly = np.array(seg).reshape(-1, 2)
            ax.plot(poly[:, 0], poly[:, 1], linewidth=2, label=f"Shape {ann['category_id']}")

ax.legend()
plt.show()